# 05 · Load and view Mario (the NES emulator)

**Goal:** see how the repo talks to the game, and learn the three moves everything else is built on: **reset**, **step**, and **save / load a state** (the "rewind" button).

You will learn
- how `MarioEnv` wraps the emulator (`src/fly_mario_fly/mario/env.py`)
- what one `step` gives back: the **screen**, the **2 KB of RAM**, and some game info
- how a state is saved and restored (this is how the tool lets you *rewind and retry*)

No MaleCNS data is needed for this notebook. Run the cells from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import _helpers as H                                   # small plotting helpers, see tutorials/_helpers.py
from fly_mario_fly.mario.env import MarioEnv, button_byte, level_ids

## 1. Start the emulator

`MarioEnv` is a thin wrapper around `gym-super-mario-bros` / `nes-py`. `reset()` starts a level and can idle for `noop` frames first
(the fly does nothing for that long, which is how the project makes runs start slightly differently).

In [ ]:
env = MarioEnv()
step = env.reset("1-1", noop=3)

print("screen:", step.frame.shape, " (height, width, RGB)")
print("RAM   :", step.ram.shape, " (the NES CPU RAM: 2048 bytes)")
print("done  :", step.done)
print("info  :", step.info)
print()
print(len(level_ids()), "levels available, e.g.", level_ids()[:6])

## 2. Look at the screen

The NES draws a 256 × 240 picture. Everything the fly will ever "see" is computed from **RAM**, not from this picture — the picture is only for us humans.

In [ ]:
H.show_frame(step.frame, title="World 1-1, frame 3")
plt.show()

## 3. Press buttons: `step`

`env.step(buttons)` advances the game by **one frame** (1/60 s). Buttons are names like `"right"`, `"A"` (jump), `"B"` (run).
Under the hood they become one byte (`button_byte`), the NES controller register.

In [ ]:
print("right      ->", button_byte(["right"]))
print("right + A  ->", button_byte(["right", "A"]))

frames = []
for i in range(60):
    step = env.step(["right"])
    if i % 20 == 19:
        frames.append((i + 1, step.frame.copy(), step.info["x_pos"]))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (i, f, x) in zip(axes, frames):
    H.show_frame(f, ax, title=f"after {i} frames of 'right'  (x = {x})")
plt.show()

`step.info` is the game's own bookkeeping (score, time, `x_pos`...). Mario's **world x** also lives in RAM, and the repo reads it from there
(next notebook). Let's keep walking and see what happens.

In [ ]:
xs = []
while not step.done:
    step = env.step(["right"])
    xs.append(step.info["x_pos"])
print("episode over after", env.frame_count, "frames, Mario reached x =", xs[-1])
print("game clock:", step.info["time"], "(the NES counts down from 400)")
H.show_frame(step.frame, title="the last frame")
plt.show()

Walking blindly to the right runs straight into the first Goomba (around x ≈ 300). That's exactly the problem the fly's senses have to solve.

## 4. Rewind: `save_state` / `load_state`

A saved state is just the **recipe** — the level plus every button press (and RAM poke) since `reset` — pickled into bytes. Loading replays it on the
deterministic emulator and gets back the identical RAM. (There is also a fast path for the newest save.) This is what makes
*rewind and retry* possible in the web tool.

Let's save at frame 60 (before the Goomba), walk into it, then rewind and jump over it instead.

In [ ]:
env.reset("1-1", noop=3)
for _ in range(60):
    env.step(["right"])
blob = env.save_state()
print("saved state:", len(blob), "bytes")

def attempt(blob, jump_at, hold=25, max_frames=160):
    """Rewind to `blob` (in a fresh emulator, as the tool does), press right, and hold A for `hold` frames starting at `jump_at`."""
    e = MarioEnv()
    e.load_state(blob)
    step = None
    for i in range(max_frames):
        step = e.step(["right", "A"] if jump_at <= i < jump_at + hold else ["right"])
        if step.done:
            break
    return step

# try 1: never jump (jump_at beyond the end)
s1 = attempt(blob, jump_at=10_000)
print("try 1, just walk:  done =", s1.done, " x =", s1.info["x_pos"], " score =", s1.info["score"])

Now the interesting part: **rewind and retry**, many times, from the same moment. The emulator is deterministic, so the only thing that changes
between tries is *when* we press A.

In [ ]:
results = []
for jump_at in range(0, 60, 4):
    s = attempt(blob, jump_at)
    results.append((jump_at, s.done, s.info["x_pos"], s.info["score"]))
    print(f"jump {jump_at:2d} frames after the saved moment ->", "died at x =" if s.done else "still alive at x =", s.info["x_pos"],
          "| score", s.info["score"])

In [ ]:
best = next(r for r in results if not r[1])
print("first jump time that survives:", best)
s = attempt(blob, best[0])
H.show_frame(s.frame, title=f"jump at +{best[0]} frames: alive, score {s.info['score']}")
plt.show()

The tries that scored 100 *stomped* the Goomba (a jump that lands on it). Too early or too late and Mario dies. Notice we never had to replay the first 60 frames by hand:
the saved state did that for us.

## Recap

- `reset` → start a level; `step(buttons)` → one frame; each step returns `frame`, `ram`, `done`, `info`.
- `save_state()` / `load_state()` → rewind. The saved state is the *recipe*, so it is small (a few hundred bytes) and exact.
- The emulator is deterministic: same buttons → same game. That is why every run in this repo has a **hash** and can be reproduced from a seed.

**Try this**
1. Change the frame at which you save (try 30, 90) and rerun the scan. Which jump times survive now?
2. Try the `"B"` button (run) together with `"right"`. How much further does Mario get in 60 frames?
3. Reset a different level (`env.reset("1-2")`) and look at its first screen.

**Next:** notebook 06 reads *meaning* out of the RAM: where Mario is, where the enemies and pipes are.